# CSE-CIC-IDS2018 — Dataset Exploration

Extended version of CIC-IDS2017 with additional attack scenarios.
Column names differ from 2017 (shorter abbreviations).

**Files:** 10 date-named CSVs (`02-14-2018.csv` through `03-02-2018.csv`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
BASE = Path('../../dataset/cse-cic-ids2018/')

## 1 · Load & Merge

In [ ]:
files = sorted(BASE.glob('*.csv'))
print(f'Files found: {len(files)}')
for f in files: print(f'  {f.name}')

dfs = []
for f in files:
    tmp = pd.read_csv(f, low_memory=False)
    tmp['source_date'] = f.stem
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
print(f'\nCombined shape: {df.shape}')
df.head(3)

## 2 · Column Comparison with CIC-IDS2017

CIC-IDS2018 uses abbreviated column names. A name-mapping step is needed to use both datasets together.

In [ ]:
# Map 2018 abbreviated names → 2017 full names (subset of key columns)
rename_map = {
    'Dst Port':          'Destination Port',
    'Tot Fwd Pkts':      'Total Fwd Packets',
    'Tot Bwd Pkts':      'Total Backward Packets',
    'TotLen Fwd Pkts':   'Total Length of Fwd Packets',
    'TotLen Bwd Pkts':   'Total Length of Bwd Packets',
    'Flow Byts/s':       'Flow Bytes/s',
    'Flow Pkts/s':       'Flow Packets/s',
    'Pkt Len Min':       'Min Packet Length',
    'Pkt Len Max':       'Max Packet Length',
    'Pkt Len Mean':      'Packet Length Mean',
    'Pkt Len Std':       'Packet Length Std',
    'Pkt Len Var':       'Packet Length Variance',
}
print('Example column renames (2018 → 2017 convention):')
for k, v in list(rename_map.items())[:5]:
    print(f'  {k!r:35} → {v!r}')
print(f'  ... ({len(rename_map)} mappings total)')

## 3 · Inf / NaN Audit

In [ ]:
num_cols = df.select_dtypes('number').columns
inf_counts = (df[num_cols] == np.inf).sum()
nan_counts = df[num_cols].isnull().sum()

print('Inf columns:', inf_counts[inf_counts > 0].to_dict())
print('NaN columns:', nan_counts[nan_counts > 0].to_dict())

## 4 · Label Distribution

In [ ]:
print(df['Label'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
df['Label'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Multi-class (log scale)'); axes[0].set_xscale('log')
df['Label'].apply(lambda x: 'Benign' if x == 'Benign' else 'Attack') \
    .value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','tomato'])
axes[1].set_title('Binary'); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 5 · Attack Distribution per Date

In [ ]:
pivot = df.groupby('source_date')['Label'].value_counts().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Attack types per capture date')
ax.tick_params(axis='x', rotation=20)
ax.legend(loc='upper right', fontsize=7)
plt.tight_layout(); plt.show()

## 6 · Feature Statistics

In [ ]:
df_stats = df[num_cols].replace([np.inf, -np.inf], np.nan)
df_stats.describe().T.round(3)

## 7 · Preprocessing Checklist

| Step | Required | Notes |
|---|---|---|
| Rename columns | Yes | Align with CIC-IDS2017 naming for cross-dataset use |
| Drop `Timestamp` column | Yes | Leaks temporal information |
| Replace Inf → NaN | Yes | Same issue as CIC-IDS2017 |
| Drop/impute NaN | Yes | After Inf replacement |
| Drop `source_date` | Yes | Metadata |
| Encode `Label` | Yes | 11 classes or binary |
| Scale numerics | Yes | MinMax or StandardScaler |
| Class imbalance | Severe | Benign traffic dominates |

## 8 · Train / Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder

drop_cols = ['source_date']
if 'Timestamp' in df.columns: drop_cols.append('Timestamp')

df_clean = df.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.drop(columns=drop_cols).dropna(subset=num_cols)

le = LabelEncoder()
y = le.fit_transform(df_clean['Label'])
X = df_clean.drop(columns=['Label'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Classes ({len(le.classes_)}): {list(le.classes_)}')

## 9 · Summary

- **Size:** ~16 M rows (10 date files)
- **Features:** 80 numeric + Timestamp (drop)
- **Same issues as CIC-IDS2017:** Inf in rate columns, severe class imbalance
- **Extra step:** column rename mapping needed for joint training with CIC-IDS2017
- **Timestamp column:** drop — it leaks the capture date directly